# FinBERT / DistilBERT Fine-Tuning on Real Financial News

**Run this on Kaggle or Google Colab with a free GPU (T4 / P100).**

## What this notebook does
1. Installs dependencies.
2. Fetches `zeroshot/twitter-financial-news-sentiment` (real, MIT-licensed, ~9.5k train / 2.4k val).
3. Fine-tunes **FinBERT** (`ProsusAI/finbert`) for 3 epochs with mixed precision (AMP).
4. Evaluates on the held-out validation split → honest OOD macro-F1.
5. Saves the model weights and `results/finetune_results.json`.
6. Tells you exactly what to do with the outputs.

## Kaggle setup (one-time)
- Create a new Notebook → set **Accelerator = GPU T4 x2** (or any GPU).
- Upload this `.ipynb` or paste each cell.
- Click **Run All**. Total time ≈ 15–20 min on T4.

## Colab setup
- `Runtime → Change runtime type → T4 GPU`.
- Click **Run All**.

In [ ]:
# ── 0. Install deps ────────────────────────────────────────────────────────
!pip install -q transformers==4.40.0 datasets accelerate scikit-learn tqdm

In [ ]:
# ── 1. Clone your repo (edit the URL to your fork if needed) ───────────────
import subprocess, os

REPO_URL = "https://github.com/hemangbhat/FinSentAnalyzer.git"
REPO_DIR = "/kaggle/working/FinSentAnalyzer"  # Colab: "/content/FinSentAnalyzer"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_DIR], check=True)

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [ ]:
# ── 2. Fetch the real dataset ──────────────────────────────────────────────
!python scripts/fetch_real_news_dataset.py

In [ ]:
# ── 3. Confirm GPU ─────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected. Change the runtime to GPU before continuing.")

In [ ]:
# ── 4. Fine-tune ───────────────────────────────────────────────────────────
# Change --model to 'distilbert' for a faster run (≈5 min/epoch on T4).
# FinBERT (ProsusAI/finbert) is the most domain-appropriate choice.
!python scripts/finetune_finbert_news.py \
    --model finbert \
    --epochs 3 \
    --batch-size 32 \
    --lr 2e-5

In [ ]:
# ── 5. Show result ─────────────────────────────────────────────────────────
import json
from pathlib import Path

result_path = Path("results/finetune_results.json")
r = json.loads(result_path.read_text())

print("\n" + "="*52)
print("In-domain fine-tuning result")
print("="*52)
print(f"Model:      {r['model']}")
print(f"Dataset:    {r['dataset']}")
print(f"Train rows: {r['n_train']}   Val rows: {r['n_val']}")
print(f"Epochs:     {r['epochs']}")
print(f"Accuracy:   {r['accuracy']:.3f}")
print(f"Macro-F1:   {r['f1_macro']:.3f}")
print()
print("Baseline comparison (TF-IDF SVM on same val set)")
print("  Accuracy  0.670   Macro-F1  0.460")
print(f"  --> Improvement: +{r['accuracy']-0.670:.3f} acc, +{r['f1_macro']-0.460:.3f} F1")

In [ ]:
# ── 6. Package outputs for download ───────────────────────────────────────
# Creates a zip with the fine-tuned weights + result JSON so you can
# download them from Kaggle/Colab and drop into models/ + results/.
import shutil

out_dir  = Path("/kaggle/working/finsight_finetuned")
out_dir.mkdir(exist_ok=True)

# Copy model weights
src_model = Path(f"models/{r['model']}_finetuned")
if src_model.exists():
    shutil.copytree(src_model, out_dir / src_model.name, dirs_exist_ok=True)
    print(f"Model weights copied to {out_dir / src_model.name}")

# Copy result JSON
shutil.copy(result_path, out_dir / "finetune_results.json")

# Zip for easy download
zip_path = shutil.make_archive(str(out_dir), "zip", out_dir)
print(f"\nDownload ready: {zip_path}")
print("\nAfter downloading, extract and copy:")
print(f"  {r['model']}_finetuned/ -> your-repo/models/{r['model']}_finetuned/")
print("  finetune_results.json   -> your-repo/results/finetune_results.json")

## After downloading the outputs

1. Copy `finbert_finetuned/` (or `distilbert_finetuned/`) into your local `models/` folder.
2. Copy `finetune_results.json` into your `results/` folder.
3. Run the generalization eval locally to confirm:
   ```bash
   python src/integrate_news.py --action generalization --model finbert
   ```
4. Refresh the model registry:
   ```bash
   python src/registry.py --update
   ```
5. The fine-tuned model now appears in the dashboard sidebar and the API
   automatically (via `get_available_models()`).
6. Update the README generalization table with the new numbers and push.

### Expected numbers (T4, 3 epochs, FinBERT)
- Accuracy ≈ 0.85–0.88
- Macro-F1 ≈ 0.79–0.83

(Achieved here without GPU: DistilBERT 1-epoch → acc 0.807 / macro-F1 0.731)